# Ingesta y vectorizacion de datos de hardware (ETL)

Este cuaderno documenta el proceso de Extraccion, Transformacion y Carga (ETL) para la creacion de una base de datos vectorial inmutable centrada en hardware de PC y videojuegos. 

El pipeline realiza las siguientes operaciones:
1. **Extraccion:** Lectura de archivos locales en formato `.json` y `.jsonl`.

2. **Transformacion:** Agrupacion estrategica de compatibilidades (lotes de 4 componentes para maximizar la densidad semantica), conversion de JSON a texto natural y fragmentacion del texto.

3. **Carga:** Vectorizacion utilizando el modelo de alta dimensionalidad `BAAI/bge-m3` acelerado por GPU, y subida por lotes a un cluster de Zilliz Cloud (Milvus).

### 1. Importacion de librerias y configuracion de credenciales
Se importan las dependencias necesarias y se cargan las variables de entorno para la conexion segura con Zilliz Cloud.

In [ ]:
import os
import json
import torch
from dotenv import load_dotenv
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus
from langchain_core.documents import Document

# Cargar credenciales
load_dotenv()
ZILLIZ_URI = os.getenv("ZILLIZ_URI")
ZILLIZ_TOKEN = os.getenv("ZILLIZ_TOKEN")

### 2. Configuracion de hardware y modelo de embeddings
Deteccion de aceleracion por hardware (CUDA) e inicialización del modelo de embeddings `BAAI/bge-m3` (1024 dimensiones) para garantizar la maxima precision en la representación semantica de los componentes.

In [ ]:
# Configurar hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de procesamiento: {DEVICE.upper()}")

# Inicializar modelo de embeddings
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32}
)

### 3. Procesamiento y agrupacion de compatibilidades
Funcion diseñada para evitar la explosion combinatoria de datos. Lee los archivos de compatibilidad y agrupa los componentes por placa base en lotes reducidos (`lote_size = 4`). Esto crea vectores muy densos que evitan la dilucion semantica y mejoran la recuperacion de contexto.

In [ ]:
def procesar_compatibilidades(directorio_datos):
    archivos_compat = [f for f in os.listdir(directorio_datos) if f in ["pangoly.jsonl", "pcpartpicker_motherboards+components.jsonl"]]
    agrupacion = {}
    
    for archivo in archivos_compat:
        ruta = os.path.join(directorio_datos, archivo)
        with open(ruta, 'r', encoding='utf-8') as f:
            for linea in f:
                if linea.strip():
                    item = json.loads(linea)
                    if "motherboard" in item and "compatible" in item:
                        mb = item["motherboard"]
                        precio_mb = item.get("motherboard_price", "N/A")
                        categoria = item.get("component_type", "otros componentes")
                        comp_name = item.get("component_name")
                        comp_price = item.get("component_price", "N/A")
                        
                        if mb not in agrupacion:
                            agrupacion[mb] = {"precio": precio_mb, "categorias": {}}
                            
                        if categoria not in agrupacion[mb]["categorias"]:
                            agrupacion[mb]["categorias"][categoria] = []
                            
                        agrupacion[mb]["categorias"][categoria].append(f"{comp_name} ({comp_price} EUR)")

    documentos_agrupados = []
    for mb, datos in agrupacion.items():
        for categoria, componentes in datos["categorias"].items():
            
            lote_size = 4 # Tamaño de lote ajustado para la maxima precision posible
            for i in range(0, len(componentes), lote_size):
                sub_lista = componentes[i:i+lote_size]
                texto = f"COMPATIBILIDAD PLACA BASE: La placa base '{mb}' (Precio estimado: {datos['precio']} EUR) es COMPATIBLE con los siguientes componentes de la categoría '{categoria}': "
                texto += ", ".join(sub_lista) + "."
                
                documentos_agrupados.append(Document(page_content=texto, metadata={"fuente": "compatibilidades_agrupadas"}))
                
    return documentos_agrupados

### 4. Transformacion de JSON a texto natural
Funcion de mapeo que convierte la estructura clave-valor de los archivos JSON restantes en parrafos de texto natural optimizados para la lectura del LLM.

In [ ]:
def transformar_json_a_texto(item, nombre_archivo):
    texto = ""
    
    if "game_name" in item and "processor" in item:
        tipo = "Requisitos Recomendados" if "recomendados" in nombre_archivo.lower() else "Requisitos Mínimos"
        texto = f"{tipo} para el videojuego '{item.get('game_name')}':\nOS: {item.get('os')}\nCPU: {item.get('processor')}\nRAM: {item.get('memory')}\nGPU: {item.get('graphics')}"
    
    elif "Name" in item and "Price" in item:
        precio = str(item.get("Price", "")).replace("Add", "").strip()
        especificaciones = [f"{k}: {str(v).replace(k, '').strip()}" for k, v in item.items() if k not in ["Name", "Price", "image_url", "Rating"] and str(v).replace(k, '').strip()]
        texto = f"COMPONENTE HARDWARE: {item.get('Name')}.\nPrecio: {precio}.\nEspecificaciones: " + " | ".join(especificaciones)
    
    elif "name" in item and "price" in item:
        texto = f"COMPONENTE HARDWARE: {item.get('name')}. Categoría: {item.get('component_type', 'N/A')}. Precio: {item.get('price')} EUR."
    
    elif "usage_percentage" in item:
        texto = f"ESTADÍSTICAS STEAM: Componente '{item.get('component')}' ({item.get('category')}) tiene uso del {item.get('usage_percentage')}."
    
    else:
        texto = "INFORMACIÓN: " + " | ".join([f"{k}: {v}" for k, v in item.items() if "url" not in k.lower() and "image" not in k.lower()])
    
    return texto.strip()

### 5. Generador de documentos estandar
Generador que itera sobre los archivos de componentes y videojuegos, procesa el texto e inyecta el JSON original como metadato oculto (limitado a 3000 caracteres) para preservar la riqueza de los datos en la base vectorial.

In [ ]:
def generador_documentos_restantes(directorio_datos):
    archivos_restantes = [f for f in os.listdir(directorio_datos) if f.endswith(('.json', '.jsonl')) and f not in ["pangoly.jsonl", "pcpartpicker_motherboards+components.jsonl"]]
    
    for archivo in archivos_restantes:
        ruta = os.path.join(directorio_datos, archivo)
        if archivo.endswith('.json'):
            with open(ruta, 'r', encoding='utf-8') as f:
                for item in json.load(f):
                    texto = transformar_json_a_texto(item, archivo)
                    yield Document(page_content=texto, metadata={"fuente": archivo, "json_data": json.dumps(item, ensure_ascii=False)[:3000]})
        
        elif archivo.endswith('.jsonl'):
            with open(ruta, 'r', encoding='utf-8') as f:
                for linea in f:
                    if linea.strip():
                        item = json.loads(linea)
                        texto = transformar_json_a_texto(item, archivo)
                        yield Document(page_content=texto, metadata={"fuente": archivo, "json_data": json.dumps(item, ensure_ascii=False)[:3000]})

### 6. Ejecucion principal: conexion, fragmentacion y carga
Bloque de control que orquesta el proceso ETL. Instancia la conexión con Zilliz Cloud, define la estrategia de fragmentacion de texto (`RecursiveCharacterTextSplitter`) y ejecuta la subida de vectores en lotes controlados para optimizar el ancho de banda y la memoria.

In [ ]:
def main():
    directorio_datos = "./data" 
    
    # 1. Procesar compatibilidades
    docs_agrupados = procesar_compatibilidades(directorio_datos)
    print(f"Documentos de compatibilidad agrupados: {len(docs_agrupados)}")
    
    # 2. Configurar conexion a Zilliz Cloud
    vectorstore = Milvus(
        embedding_function=embeddings,
        connection_args={"uri": ZILLIZ_URI, "token": ZILLIZ_TOKEN, "secure": True},
        collection_name="ai_data", 
        drop_old=True, 
        auto_id=True
    )
    
    # 3. Configurar fragmentación
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
    
    # 4. Subida de ficheros de compatibilidades
    chunks_agrupados = text_splitter.split_documents(docs_agrupados)
    BATCH_SIZE = 1500
    for i in tqdm(range(0, len(chunks_agrupados), BATCH_SIZE), desc="Subiendo de ficheros de compatibilidades"):
        vectorstore.add_documents(documents=chunks_agrupados[i:i + BATCH_SIZE])
        
    # 5. Subida Fase 2 (Resto de componentes y metadatos)
    batch_actual = []
    for doc in tqdm(generador_documentos_restantes(directorio_datos), desc="Procesando componentes/juegos"):
        chunks = text_splitter.split_documents([doc])
        batch_actual.extend(chunks)
        
        if len(batch_actual) >= BATCH_SIZE:
            vectorstore.add_documents(documents=batch_actual)
            batch_actual = []
            
    if batch_actual:
        vectorstore.add_documents(documents=batch_actual)

    print("\nProceso de ingestion completado exitosamente.")

if __name__ == "__main__":
    main()